# 01 코드 해설 — 사진 한 장을 모델에 넣으면 어떤 일이 일어날까요?

[01번 실습 노트북](../notebooks/01_gpu_inference.ipynb)의 모든 코드 셀을 순서대로 풀이한 학생용 읽기 자료입니다. 코드가 낯설다면 먼저 이 문서를 읽고 원본 실습으로 돌아가세요.

**이 노트북에서 실행되는 것은 Python 기본 기능으로 만든 작은 CPU 예제 4개뿐입니다.** 라이브러리 설치, 모델 다운로드, 파일 저장, Colab GPU 연결은 하지 않습니다. 긴 원본 코드는 모두 Markdown의 읽기용 코드 상자입니다. 전체 실행을 눌러도 원본 GPU 코드는 실행되지 않습니다.

원본 01번은 준비된 모델·데이터와 Colab GPU가 필요합니다. 두 핵심 함수의 완성 코드가 포함되어 있으므로 위에서부터 실행할 수 있습니다. 여기서는 각 코드 줄의 역할과 실행 결과를 풀어 설명합니다.

읽을 순서는 **준비 → 파일 확인 → 모델 읽기 → pipeline 예제 → 두 함수 만들기 → 결과 확인**입니다. 원본의 '몇 번째 셀' 표시는 Markdown 설명 셀도 포함한 위치이며 화면의 `In [ ]` 실행 횟수와는 다릅니다.

### 사진과 숫자는 이렇게 이동합니다

| 단계 | 데이터의 모습 | 하는 일 |
|---|---|---|
| 원본 사진 | RGB 사진 한 장, 32×32 | 어떤 물체인지 살펴봅니다. |
| 전처리 결과 | `(1, 3, 224, 224)` 텐서 | 사진 수·색 채널·세로·가로 순서로 모델 입력을 만듭니다. |
| 모델 출력 | `(1, 1000)` logits | 원래 모델이 아는 ImageNet 1,000개 항목의 점수를 계산합니다. |
| softmax 결과 | `(1, 1000)`, 한 장을 꺼내면 `(1000,)` | 점수 1,000개를 합계가 약 1인 값으로 바꿉니다. |
| 상위 후보 | 이름과 점수 5쌍 | 사람이 읽기 쉬운 예측 목록으로 정리합니다. |

`1000`은 모델의 분류 항목 수입니다. 실습 데이터의 정답 종류인 bottle·bowl·can·cup·plate 다섯 개와 혼동하지 마세요. **추론**은 이미 배운 모델로 답을 계산하는 과정이고 **학습**은 모델 안의 가중치를 바꾸는 과정입니다. 01번은 추론만 합니다.

## 1. 이전 실행의 결과를 비우기

원본의 2번째 셀

아래 회색 상자는 원본 코드의 읽기용 복사본입니다. 이 해설 노트북에서는 실행되지 않습니다.

```python
INFERENCE_CHECKS = {"01-infer": False, "01-topk": False}
for name in ("infer_probabilities", "topk_predictions", "probabilities", "predictions"):
    globals().pop(name, None)
```

### 무엇을 준비하나요?

- `INFERENCE_CHECKS`는 문제 두 개의 확인 상태를 담는 **딕셔너리**입니다. 이름표인 `"01-infer"`, `"01-topk"` 각각에 `False`를 넣어 아직 통과하지 않았다고 표시합니다.
- `True`와 `False`는 참·거짓을 나타내는 Python 값입니다. 문자열 `"False"`와는 다르므로 따옴표를 붙이지 않습니다.
- `for name in (...)`은 괄호 안의 이름을 하나씩 꺼내 같은 작업을 반복합니다. 다음 줄의 들여쓰기는 반복할 범위를 표시합니다.
- `globals()`는 현재 노트북에서 전역 이름으로 등록한 값과 함수를 찾아볼 수 있는 공간입니다.
- `.pop(name, None)`은 그 이름이 있으면 지우고 없어도 오류 없이 넘어갑니다. 두 번째 인자 `None`이 '없을 때 사용할 값'입니다.
- 노트북은 위 셀에서 만든 변수를 기억합니다. 예전에 실행한 함수가 남으면 수정 전 코드와 현재 코드를 혼동할 수 있어 초기화합니다. 이어지는 완성 함수 셀을 실행해 현재 정의를 사용합니다.
- 셀을 실행해도 화면에 아무것도 출력되지 않는 것이 정상입니다. 이후 셀이 사용할 상태만 바뀝니다.
- 실습 도중 이 셀을 다시 실행했다면 두 함수가 사라집니다. 원본의 함수 작성 셀과 확인 셀을 다시 실행하세요. 이 초기화는 모델 가중치를 학습시키는 코드가 아닙니다.

## 2. 도구를 불러오고 GPU를 확인하기

원본의 4번째 셀

아래 회색 상자는 원본 코드의 읽기용 복사본입니다. 이 해설 노트북에서는 실행되지 않습니다.

```python
TRAINING_ALLOWED = False
import os
import sys
import json
import hashlib
import random
from pathlib import Path
from importlib.metadata import version

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from IPython import get_ipython
import torch
from transformers import AutoImageProcessor, AutoModelForImageClassification

ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("matplotlib", "inline")
if not torch.cuda.is_available():
    raise RuntimeError("GPU가 없습니다. Colab CLI GPU 세션으로 실행하세요.")
device = torch.device("cuda")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)
plt.rcParams.update({"figure.dpi": 110, "font.size": 11})
print("Python:", sys.executable)
print("GPU:", torch.cuda.get_device_name(0))
print({name: version(name) for name in ("torch", "transformers", "huggingface-hub")})
```

### 준비 단계가 긴 이유

1. `TRAINING_ALLOWED = False`는 이번 단계가 추론임을 표시하는 변수입니다. 이 이름만으로 모든 학습을 막는 것은 아닙니다. 뒤에서 추론 모드와 가중치 검사를 함께 사용합니다.
2. `import`는 도구를 불러옵니다. `os`는 환경 설정, `sys`는 Python 정보, `json`은 기록 파일, `hashlib`는 파일 지문, `random`은 난수, `Path`는 경로, `version`은 설치 버전 확인에 쓰입니다.
3. `HF_HUB_OFFLINE`, `TRANSFORMERS_OFFLINE`의 문자열 값 `"1"`은 이미 받은 로컬 파일을 쓰겠다는 설정입니다. 파일이 없을 때 인터넷에서 대신 받지 않으므로 먼저 00번 준비를 마쳐야 합니다.
4. `CUBLAS_WORKSPACE_CONFIG`는 GPU 계산의 재현성 설정입니다. GPU 관련 라이브러리를 본격적으로 쓰기 전에 지정합니다.
5. `numpy`는 이미지 숫자 배열, `PIL.Image`는 사진 객체, `matplotlib.pyplot`은 그래프, `torch`는 텐서·모델 계산을 맡습니다. `as np`, `as plt`는 긴 도구 이름을 짧게 부르는 별명입니다.
6. `AutoImageProcessor`와 `AutoModelForImageClassification`은 모델에 맞는 전처리기와 이미지 분류 모델을 고르는 Transformers 도구입니다. 이 줄에서는 이름만 가져오고 실제 모델 파일은 뒤에서 읽습니다.
7. `get_ipython()`은 노트북 실행 환경을 찾습니다. 환경이 있을 때만 `matplotlib inline`을 설정해 그래프를 셀 아래에 표시합니다. `is not None`은 '실제로 찾았는가'를 확인합니다.
8. `torch.cuda.is_available()`이 거짓이면 `raise RuntimeError(...)`로 멈춥니다. **Codespaces CPU에서 원본 셀을 실행하면 여기서 멈추는 것이 의도된 동작**입니다. 모델 추론은 연결된 Colab GPU 세션에서 합니다.
9. `device = torch.device("cuda")`는 앞으로 쓸 GPU 장치를 가리킵니다. CPU 메모리의 입력과 GPU의 모델은 그대로 함께 계산할 수 없으므로 나중에 입력도 같은 장치로 옮깁니다.
10. `SEED = 42`와 여러 `.seed()` 호출은 서로 다른 도구의 난수 출발점을 맞춥니다. `benchmark = False`와 `use_deterministic_algorithms(True)`도 반복 실행의 일관성을 돕지만 다른 하드웨어·버전까지 완전히 같은 결과를 보장하지는 않습니다.
11. `plt.rcParams.update(...)`는 그림의 기본 해상도와 글자 크기를 정합니다. 마지막 `print()` 세 줄은 Python 실행 위치, GPU 이름, 라이브러리 버전을 보여 줍니다.
12. `{name: version(name) for name in (...)}`은 라이브러리 이름과 버전을 짝지어 딕셔너리로 만드는 문법입니다. 오류가 나면 00번 준비, Colab 설치 단계, 현재 실행 위치를 확인하세요.

## 3. 모델 파일의 위치와 지문 확인하기

원본의 5번째 셀

아래 회색 상자는 원본 코드의 읽기용 복사본입니다. 이 해설 노트북에서는 실행되지 않습니다.

```python
candidates = [Path.cwd(), *Path.cwd().parents, Path("/content/vision-ai")]
ROOT = next((p for p in candidates if (p / ".vision-lab-root").is_file()), None)
if ROOT is None:
    raise FileNotFoundError(".vision-lab-root와 실습 파일을 먼저 업로드하세요.")
MODEL_ID = 'facebook/deit-tiny-patch16-224'
MODEL_REVISION = 'b3428f18dcc7b543470d07f14b4a4157815d1880'
MODEL_DIR = ROOT / "hf_colab_gpu/models/deit-tiny"
DATA_DIR = ROOT / "data/prepared"
OUTPUT_DIR = ROOT / "hf_colab_gpu/results/gpu"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

download_manifest = json.loads((MODEL_DIR / "download_manifest.json").read_text())
assert download_manifest["model_id"] == MODEL_ID
assert download_manifest["revision"] == MODEL_REVISION
for name in ("config.json", "preprocessor_config.json", "pytorch_model.bin"):
    assert sha256_file(MODEL_DIR / name) == download_manifest["files"][name], name
print("HF CLI 다운로드 파일 확인:", MODEL_ID, MODEL_REVISION[:12])
```

### 어떤 파일을 읽는지 먼저 고정합니다

1. `Path.cwd()`는 현재 폴더입니다. `.parents`는 상위 폴더들이며 `*`는 이 목록을 후보 목록 안에 펼쳐 넣습니다. 마지막 후보는 `/content/vision-ai`입니다.
2. `next((... for ... if ...), None)`은 조건에 맞는 첫 폴더를 찾습니다. `.vision-lab-root` 표식 파일이 있는 폴더를 프로젝트 시작점 `ROOT`로 삼습니다.
3. 시작점을 찾지 못하면 `FileNotFoundError`로 멈춥니다. 이름만 비슷한 빈 폴더를 만드는 대신 실습 파일과 표식 파일을 함께 올렸는지 확인합니다.
4. `MODEL_ID`는 Hub 모델 이름, `MODEL_REVISION`은 사용할 모델 버전을 가리키는 커밋입니다. 둘을 함께 기록해야 어느 모델 파일을 썼는지 분명해집니다.
5. `ROOT / "..."`의 `/`는 `Path` 객체에서 경로를 이어 붙이는 연산입니다. 숫자를 나누는 계산이 아닙니다.
6. `MODEL_DIR`, `DATA_DIR`, `OUTPUT_DIR`는 모델, 준비 데이터, 결과 저장 폴더입니다. `.mkdir(parents=True, exist_ok=True)`는 필요한 상위 폴더까지 만들고 이미 있어도 넘어갑니다.
7. `def sha256_file(path):`는 파일 바이트를 읽고 SHA-256 지문 문자열로 돌려주는 함수 정의입니다. `return`한 값은 뒤의 비교에 쓸 수 있습니다. 화면에 보여 주는 `print`와 다릅니다.
8. `json.loads(...read_text())`는 JSON 파일의 글을 Python 딕셔너리로 바꿉니다. `download_manifest["model_id"]`처럼 이름표로 원하는 값을 꺼냅니다.
9. `assert`는 조건이 틀리면 실행을 멈춥니다. 모델 이름·버전과 세 파일의 지문이 다운로드 기록과 같은지 확인하므로 다른 파일을 섞은 상태로 계속 진행하지 않습니다.
10. 파일 지문 일치는 **기록한 파일과 같은 파일인지**를 확인하는 것입니다. 모델이 정확하거나 파일 자체가 안전하다는 보증은 아닙니다. `MODEL_REVISION[:12]`는 긴 커밋의 앞 12글자만 출력하는 슬라이싱입니다.

## 4. 데이터가 섞이거나 손상되지 않았는지 확인하기

원본의 7번째 셀

아래 회색 상자는 원본 코드의 읽기용 복사본입니다. 이 해설 노트북에서는 실행되지 않습니다.

```python
manifest_file = DATA_DIR / "manifest.json"
manifest = json.loads(manifest_file.read_text(encoding="utf-8"))
classes = manifest["classes"]
if len(classes) < 2 or len(set(classes)) != len(classes):
    raise ValueError("클래스 목록을 확인하세요.")
identity = {k: manifest[k] for k in ("classes", "splits", "seed", "preprocess")}
fingerprint = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()
assert fingerprint == manifest["dataset_sha256"]
splits, seen_ids = {}, set()
for split in ("train", "validation", "test"):
    record = manifest["splits"][split]
    assert record["file"] == f"{split}.npz"
    file = DATA_DIR / record["file"]
    assert sha256_file(file) == record["sha256"], split
    with np.load(file, allow_pickle=False) as stored:
        images, labels, ids = stored["images"], stored["labels"], stored["ids"]
    assert images.dtype == np.uint8 and images.ndim == 4 and images.shape[-1] == 3
    assert labels.ndim == 1 and np.issubdtype(labels.dtype, np.integer)
    assert len(images) == len(labels) == len(ids) == record["count"] > 0
    assert labels.min() >= 0 and labels.max() < len(classes)
    assert len(set(ids)) == len(ids) and not seen_ids.intersection(ids)
    seen_ids.update(ids)
    splits[split] = {"images": images, "labels": labels, "ids": ids}
    print(split, len(labels), "images")
print("클래스 순서:", classes)
```

### 사진·정답·사진 ID는 같은 순서로 움직입니다

1. `manifest.json`은 데이터 설명서입니다. `classes`는 정답 번호와 이름의 순서를 담으며 이 실습에서는 다섯 종류를 사용합니다.
2. `len(classes)`는 항목 수, `set(classes)`는 중복을 없앤 집합입니다. 두 길이를 비교해 이름이 중복되지 않았는지 봅니다.
3. `identity = {k: manifest[k] for k in (...)}`는 분할·시드·전처리 등 필요한 부분을 추립니다. `sort_keys=True`로 이름 순서를 고정한 뒤 지문을 계산해 준비할 때의 기록과 비교합니다.
4. `splits = {}`에는 읽은 데이터 묶음을 넣고 `seen_ids = set()`에는 지금까지 본 사진 ID를 모읍니다. 쉼표를 사용해 두 변수를 한 줄에서 초기화한 것입니다.
5. `for split in ("train", "validation", "test")`는 학습·검증·최종 평가 파일을 차례로 읽습니다. 01번에서 파일 세 개를 확인하더라도 학습을 실행하는 것은 아닙니다.
6. 파일 이름과 지문을 먼저 검사합니다. `f"{split}.npz"`는 `split` 값에 따라 `train.npz` 같은 문자열을 만드는 f-string입니다.
7. `with np.load(..., allow_pickle=False) as stored:`는 숫자 배열 묶음을 열고 사용 후 닫습니다. 임의의 Python 객체를 복원하는 pickle 기능은 허용하지 않습니다.
8. `images`, `labels`, `ids`는 사진 숫자 배열, 정답 번호, 사진 식별자입니다. 같은 위치의 항목 세 개가 한 사진에 해당해야 하므로 길이가 모두 같은지 검사합니다.
9. `images.dtype == np.uint8`은 각 색 값이 8비트 부호 없는 정수인지 확인합니다. `ndim == 4`와 `shape[-1] == 3`은 `(사진 수, 세로, 가로, RGB)` 모양인지 검사합니다. `-1`은 마지막 축입니다.
10. 정답은 1차원 정수 배열이며 `0`부터 `len(classes)-1`까지여야 합니다. 다섯 종류면 정답 번호는 `0, 1, 2, 3, 4`입니다.
11. `seen_ids.intersection(ids)`는 앞서 읽은 묶음과 겹치는 사진 ID를 찾습니다. 겹침이 없어야 같은 사진을 학습과 평가 양쪽에 쓰는 문제를 막을 수 있습니다.
12. 출력은 각 묶음의 사진 수와 클래스 순서입니다. 기본 데이터는 학습 500장·검증 100장·최종 평가 200장입니다. 이 수치는 데이터 구성이고 모델의 정확도가 아닙니다.

### CPU로 살펴보기 — 작은 RGB 배열로 차원 읽기

색 세 개가 한 점을 만들고 점이 모여 한 행, 행이 모여 한 사진을 만듭니다. 아래 2×2 예시는 순서를 읽는 연습입니다. 원본은 데이터의 `(N, H, W, C)` 배열에서 한 사진을 꺼낸 뒤 전처리기가 모델용 `(N, C, H, W)`로 바꾸고 224×224 입력으로 만듭니다.

아래는 **개념을 설명하는 작은 예시**입니다. 실제 사진이나 모델의 추론 결과가 아닙니다.

In [1]:
# 설명용 숫자로 만든 2×2 RGB 사진입니다. 실제 모델 입력은 아닙니다.
small_image = [
    [[255, 0, 0], [0, 255, 0]],
    [[0, 0, 255], [255, 255, 255]],
]
batch = [small_image]
data_shape = (len(batch), len(batch[0]), len(batch[0][0]), len(batch[0][0][0]))
print("데이터 순서 (N, H, W, C):", data_shape)
print("첫 사진의 첫 행, 첫 점:", batch[0][0][0])
print("이 점의 R 값:", batch[0][0][0][0])
assert data_shape == (1, 2, 2, 3)

데이터 순서 (N, H, W, C): (1, 2, 2, 3)
첫 사진의 첫 행, 첫 점: [255, 0, 0]
이 점의 R 값: 255


## 5. 전처리 도구와 모델을 실제로 읽기

원본의 9번째 셀

아래 회색 상자는 원본 코드의 읽기용 복사본입니다. 이 해설 노트북에서는 실행되지 않습니다.

```python
processor = AutoImageProcessor.from_pretrained(
    MODEL_DIR, local_files_only=True, use_fast=False,
)
model = AutoModelForImageClassification.from_pretrained(
    MODEL_DIR, local_files_only=True, use_safetensors=False, weights_only=True,
    attn_implementation="eager",
).to(device)
print(type(model).__name__, "· 원래 분류 수:", model.config.num_labels)
print("전체 파라미터:", f"{sum(p.numel() for p in model.parameters()):,}")
```

### 같은 사진을 모델이 기대하는 형식으로 맞춥니다

1. `processor`는 사진을 모델 입력으로 바꾸는 도구입니다. 모델이 학습할 때 사용한 크기 조정·정규화 설정을 `MODEL_DIR`에서 읽습니다. 정규화는 색 숫자의 범위를 모델에 맞추는 과정입니다.
2. `Auto...from_pretrained(...)`는 저장된 설정을 보고 알맞은 종류의 도구나 모델을 만듭니다. `pretrained`는 이미 학습된 가중치를 읽는다는 뜻이지 지금 학습한다는 뜻이 아닙니다.
3. `local_files_only=True`는 로컬 파일만 읽도록 명시합니다. 준비 파일이 없으면 다운로드로 넘어가지 않고 오류를 냅니다.
4. `use_fast=False`는 이 실습에서 사용할 전처리 구현을 고정합니다. 이름만 보고 '빼면 무조건 더 좋다'고 해석하지 말고 비교할 때 설정을 유지합니다.
5. `use_safetensors=False`는 원본 모델의 `pytorch_model.bin`을 사용하기 위한 설정입니다. `weights_only=True`는 가중치 로딩 시 복원할 객체를 제한하지만 모든 위험을 없애 주는 장치는 아닙니다.
6. `attn_implementation="eager"`는 모델 내부 attention 계산 구현을 지정합니다. 초보 실습에서는 계산 방식의 세부 차이보다 제공한 설정을 유지하는 것이 우선입니다.
7. `.to(device)`는 모델 가중치를 GPU로 옮깁니다. 전처리한 입력도 같은 GPU에 있어야 모델과 함께 계산할 수 있습니다.
8. `model.config.num_labels`는 원래 모델의 출력 항목 수 1,000을 알려 줍니다. 데이터 정답 다섯 종류로 바꾸는 작업은 02번에서 합니다.
9. `model.parameters()`는 모델의 가중치 묶음을 순회하고 `p.numel()`은 각 묶음의 숫자 개수를 셉니다. `sum(...)`으로 전체 파라미터 수를 구합니다.
10. `f"{...:,}"`는 큰 정수에 세 자리마다 쉼표를 넣는 출력 형식입니다. 정상 출력은 모델 종류·분류 수·전체 파라미터 수이며 아직 사진 예측은 하지 않았습니다.

## 6. pipeline으로 전체 추론 흐름 먼저 보기

원본의 11번째 셀

아래 회색 상자는 원본 코드의 읽기용 복사본입니다. 이 해설 노트북에서는 실행되지 않습니다.

```python
from transformers import pipeline

PIPELINE_TOP_K = 5
pipeline_class = classes.index("cup")
pipeline_index = int(np.flatnonzero(splits["test"]["labels"] == pipeline_class)[0])
pipeline_image = Image.fromarray(splits["test"]["images"][pipeline_index]).convert("RGB")
model.eval()
image_classifier = pipeline(
    "image-classification", model=model, image_processor=processor,
    framework="pt", device=device,
)
pipeline_predictions = image_classifier(pipeline_image, top_k=PIPELINE_TOP_K)
print("실습 데이터 정답:", classes[pipeline_class])
print("pipeline 출력: label(이름), score(점수)")
for item in pipeline_predictions:
    print(f"{item['score']:6.2%}  {item['label']}")
```

### pipeline은 무엇을 묶어 주나요?

1. `from transformers import pipeline`은 추론에 자주 쓰는 절차를 묶어 주는 도구를 불러옵니다. 이미지 분류에서는 **사진 전처리 → 모델 계산 → 이름과 점수 정리**를 연결합니다.
2. `PIPELINE_TOP_K = 5`는 보여 줄 후보 수입니다. `1`이나 `3`으로 바꿀 수 있으며 모델이 아는 분류 항목 수 자체는 1,000개로 유지됩니다.
3. `classes.index("cup")`은 다섯 정답 종류 중 cup의 번호를 찾습니다. 이 번호는 **사진을 고르는 용도**이고 모델의 예측 이름을 찾는 번호가 아닙니다.
4. `splits["test"]["labels"] == pipeline_class`는 각 사진이 cup인지 비교합니다. `np.flatnonzero(...)`는 맞는 위치를 모으고 `[0]`은 그중 첫 번째 위치를 고릅니다.
5. `Image.fromarray(...).convert("RGB")`는 숫자 배열을 PIL 이미지로 바꾸고 빨강·초록·파랑 세 채널 형식을 맞춥니다.
6. `model.eval()`은 모델을 평가 동작으로 바꿉니다. 학습 때 사용하는 일부 동작을 평가에 맞게 바꾸는 설정이며 이것만으로 기울기 추적을 끄는 것은 아닙니다.
7. `pipeline("image-classification", model=model, image_processor=processor, ...)`은 **이미 로컬 파일에서 읽은 객체**를 재사용합니다. `framework="pt"`는 PyTorch이고 `device`는 계산할 GPU입니다.
8. `image_classifier(...)`가 사진을 받아 결과를 돌려줍니다. 반환값은 `label`과 `score`를 가진 딕셔너리들의 목록입니다. `for item in ...`은 그 목록을 한 항목씩 출력합니다.
9. `{item['score']:6.2%}`는 0~1 점수를 소수점 둘째 자리까지 퍼센트로 보여 줍니다. 계산에 사용하는 값은 그대로이며 화면 표시만 바뀝니다.
10. 이 점수는 해당 사진에서 모델이 부여한 분류 점수입니다. '사진 여러 장 중 맞힌 비율'인 **정확도**와 다르며 높은 점수라고 정답을 보장하지 않습니다.
11. 같은 사진·모델에서 `top_k`만 바꾸면 보이는 후보 수만 달라집니다. 1위만 표시한다고 그 점수가 100%로 바뀌지는 않습니다.
12. 이 셀은 00번의 HF CLI 다운로드 결과를 사용하므로 추가 다운로드가 없습니다. [별도 Hub 추론 예제](../pipeline_inference.py)는 `model=MODEL_ID, revision=MODEL_REVISION`처럼 Hub ID를 주어 처음에 다운로드하는 방식입니다. [셸 시연 안내](../pipeline-guide.md)는 그 별도 예제를 Colab에서 실행하는 순서를 설명합니다.

pipeline 예제와 뒤의 두 함수는 같은 처리 흐름을 다른 수준에서 보여 줍니다. 다음 단계에서는 제공된 추론 함수와 Top-k 함수를 읽고 각각의 확인 셀을 실행합니다.

## 7. 추론 함수에 넣을 사진 고르기

원본의 14번째 셀

아래 회색 상자는 원본 코드의 읽기용 복사본입니다. 이 해설 노트북에서는 실행되지 않습니다.

```python
# 먼저 그대로 실행하고, 성공한 뒤 다른 종류나 사진 번호를 선택해 보세요.
SAMPLE_CLASS = "cup"
SAMPLE_OFFSET = 0
INFERENCE_CHECKS.update({"01-infer": False, "01-topk": False})
for name in ("probabilities", "predictions"):
    globals().pop(name, None)
assert SAMPLE_CLASS in classes, f"선택 가능한 종류: {classes}"
target_class = classes.index(SAMPLE_CLASS)
sample_indices = np.flatnonzero(splits["test"]["labels"] == target_class)
assert isinstance(SAMPLE_OFFSET, int) and 0 <= SAMPLE_OFFSET < len(sample_indices)
image_index = int(sample_indices[SAMPLE_OFFSET])
image = Image.fromarray(splits["test"]["images"][image_index]).convert("RGB")

def model_fingerprint(model):
    digest = hashlib.sha256()
    for name, value in model.state_dict().items():
        digest.update(name.encode())
        digest.update(value.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()

print("사진의 실제 종류:", SAMPLE_CLASS, "· 같은 종류의 사진 번호:", SAMPLE_OFFSET)
print("입력 형식:", image.mode, "· 원본 크기:", image.size)
```

### 바꿔 볼 입력은 두 개입니다

1. `SAMPLE_CLASS = "cup"`은 보고 싶은 정답 종류입니다. 기본 다섯 종류 중 하나로 바꿉니다. 따옴표 없는 `cup`은 문자열이 아닌 변수 이름으로 해석됩니다.
2. `SAMPLE_OFFSET = 0`은 같은 종류 사진 중 몇 번째인지 뜻합니다. Python 순서는 0부터 시작하므로 `0`이 첫 장, `1`이 둘째 장입니다.
3. 입력 사진을 바꾸면 이전 결과를 그대로 사용할 수 없습니다. 확인 상태를 다시 `False`로 만들고 이전 확률·예측 변수를 지웁니다.
4. `assert SAMPLE_CLASS in classes`는 선택한 이름이 목록에 있는지 확인합니다. 맞지 않으면 오류 메시지에 선택 가능한 종류를 보여 줍니다.
5. `target_class`는 정답 번호이고 `sample_indices`는 그 정답에 해당하는 사진 위치 목록입니다. 정답 번호와 전체 데이터 안의 사진 위치는 다른 숫자입니다.
6. `isinstance(SAMPLE_OFFSET, int)`는 정수인지, 이어지는 조건은 범위 안인지 검사합니다. `"1"`처럼 따옴표로 묶으면 숫자 대신 문자열이 됩니다.
7. `image_index = int(...)`로 한 장의 위치를 고르고 `Image.fromarray(...).convert("RGB")`로 함수에 넘길 `image`를 만듭니다. 화면에는 원본 사진 형식과 32×32 크기가 출력됩니다.
8. `model_fingerprint`는 모델 상태 전체의 이름과 숫자를 지문으로 만드는 함수입니다. `model.state_dict()`는 가중치 등을 이름과 값으로 꺼낼 수 있는 사전 형태의 상태 묶음입니다.
9. `value.detach().cpu().contiguous().numpy().tobytes()`는 계산 그래프에서 분리한 값을 CPU로 옮겨 연속된 숫자 배열·바이트로 읽습니다. GPU 가중치를 여기서 학습하거나 새 값으로 바꾸지 않습니다.
10. 이 지문은 제공 함수 실행 전후의 가중치가 같은지 검사할 때 사용합니다. 출력이 바뀌었는지와 가중치가 바뀌었는지는 별개입니다. 사진만 바꿔도 출력은 달라질 수 있습니다.

## 8. 완성 함수 1 — 이미지에서 확률까지

원본의 16번째 셀

원본에 제공된 함수입니다. 아래 해설과 함께 코드를 읽고 원본 노트북에서 실행합니다.

```python
def infer_probabilities(model, processor, image, device):
    # 전처리한 입력을 모델과 같은 장치로 옮깁니다.
    inputs = {
        name: tensor.to(device)
        for name, tensor in processor(images=image, return_tensors="pt").items()
    }
    model.eval()
    with torch.inference_mode():
        logits = model(**inputs).logits
        return logits.softmax(dim=-1)[0]
```

### 완성된 추론 함수를 순서대로 읽기

1. `def infer_probabilities(model, processor, image, device)`는 네 입력을 받는 함수를 정의합니다. 이 셀을 실행하는 순간 사진을 추론하는 것은 아니며, 다음 확인 셀이 함수를 호출합니다.
2. `processor(images=image, return_tensors="pt")`는 RGB 사진을 PyTorch 텐서로 바꿉니다. 이 모델의 사진 한 장은 주 입력 `pixel_values`의 모양이 `(1, 3, 224, 224)`가 됩니다. 원본 사진의 32×32 크기와 구분하세요.
3. `.items()`는 전처리 결과의 이름과 텐서를 짝으로 꺼냅니다. `name: tensor.to(device)`를 반복해 모든 입력 텐서를 모델과 같은 GPU에 둡니다. 결과는 이름으로 값을 찾는 딕셔너리 `inputs`입니다.
4. `model.eval()`은 모델을 평가 동작으로 바꿉니다. 학습 때 사용하는 일부 동작을 평가에 맞추지만, 이것만으로 기울기 추적을 끄지는 않습니다.
5. `with torch.inference_mode():`는 그 안에서 학습용 기울기 기록을 하지 않도록 합니다. 평가 모드와 역할이 달라 두 설정을 함께 사용합니다.
6. `model(**inputs)`의 `**`는 딕셔너리를 이름 있는 인자로 펼칩니다. `pixel_values` 텐서를 모델에 전달하고, 출력에서 `.logits`를 꺼냅니다.
7. `logits`는 `(1, 1000)` 모양의 원점수입니다. 음수가 있거나 합계가 1이 아니어도 이 단계에서는 정상입니다.
8. `.softmax(dim=-1)`은 마지막 축의 1,000개 점수를 합계가 약 1인 값으로 바꿉니다. `[0]`은 사진 한 장의 행만 꺼내 `(1000,)` 모양으로 만듭니다.
9. `return`은 GPU의 1차원 텐서를 호출한 곳으로 돌려줍니다. 출력만 하는 `print`와 다르며, 여기서는 이름 목록이나 퍼센트 문자열로 바꾸지 않습니다.
10. 아래 확인 셀은 모델을 실제로 호출했는지, 반환값의 모양·합계·장치가 맞는지, 가중치 값이 그대로인지 검사합니다. 특정 사진을 정답으로 맞혔다는 보증은 아닙니다.

**다르게 살펴보기:** 원본의 사진 종류나 사진 번호를 바꾸고 어떤 줄이 새 사진을 처리하는지 따라가 보세요. AI에게는 완성 코드를 보여 주고 “전처리, GPU 이동, 모델 호출, 결과 반환이 각각 어느 줄인지 설명해 줘”라고 요청할 수 있습니다.

### CPU로 살펴보기 — 합계가 1인 값으로 바꾸기

softmax의 역할만 숫자 세 개로 살펴봅니다. 원본 모델은 세 개가 아니라 1,000개 점수를 냅니다. 가장 큰 점수를 먼저 빼면 큰 수의 지수 계산을 피하면서 같은 softmax 비율을 구할 수 있습니다.

아래는 **개념을 설명하는 작은 예시**입니다. 실제 사진이나 모델의 추론 결과가 아닙니다.

In [2]:
import math

# 개념 설명용 점수입니다. 실제 모델 출력이 아닙니다.
toy_logits = [2.0, 1.0, 0.0]
shift = max(toy_logits)
weights = [math.exp(value - shift) for value in toy_logits]
total = sum(weights)
toy_probabilities = [weight / total for weight in weights]
print("softmax 값:", [round(value, 4) for value in toy_probabilities])
print("합계:", round(sum(toy_probabilities), 6))
print("두 후보만 표시했을 때 합계:", round(sum(toy_probabilities[:2]), 6))
assert abs(sum(toy_probabilities) - 1.0) < 1e-12
assert sum(toy_probabilities[:2]) < 1.0

softmax 값: [0.6652, 0.2447, 0.09]
합계: 1.0
두 후보만 표시했을 때 합계: 0.909969


## 9. 실습 1 확인 셀은 무엇을 검사하나요?

원본의 17번째 셀

아래 회색 상자는 원본 코드의 읽기용 복사본입니다. 이 해설 노트북에서는 실행되지 않습니다.

```python
INFERENCE_CHECKS.update({"01-infer": False, "01-topk": False})
globals().pop("probabilities", None)
globals().pop("predictions", None)
if not callable(globals().get("infer_probabilities")):
    raise RuntimeError("앞의 infer_probabilities 함수 셀을 먼저 실행하세요.")
weights_before = model_fingerprint(model)
model.train()  # 제공된 함수가 평가 모드로 전환하는지도 확인합니다.
forward_calls = []
def record_forward(module, args, output):
    forward_calls.append((output.logits.detach().clone(), torch.is_grad_enabled(),
                          torch.is_inference_mode_enabled(), module.training))
hook = model.register_forward_hook(record_forward)
try:
    candidate = infer_probabilities(model, processor, image, device)
finally:
    hook.remove()
assert forward_calls, "확률을 직접 만들지 말고 제공된 모델을 호출하세요."
assert all(not grad and inference and not training for _, grad, inference, training in forward_calls), "평가 모드와 추론 모드 안에서 모델을 실행하세요."
assert isinstance(candidate, torch.Tensor), "반환값은 PyTorch 텐서여야 합니다."
assert candidate.ndim == 1 and len(candidate) == model.config.num_labels, "1차원 확률을 반환하세요."
assert candidate.device == next(model.parameters()).device, "확률도 모델과 같은 GPU에 있어야 합니다."
assert not candidate.requires_grad, "추론 모드를 사용하세요."
assert not model.training, "모델을 평가 모드로 전환하세요."
assert torch.isfinite(candidate).all().item(), "확률에 NaN이나 무한대가 있습니다."
assert ((candidate >= 0) & (candidate <= 1)).all().item(), "확률 범위는 0~1입니다."
assert abs(float(candidate.sum()) - 1.0) < 1e-5, "softmax와 확률 합계를 확인하세요."
torch.testing.assert_close(candidate, forward_calls[-1][0].softmax(dim=-1)[0])
assert weights_before == model_fingerprint(model), "가중치가 바뀌었습니다. 모델 불러오기 셀부터 다시 실행하세요."
probabilities = candidate
INFERENCE_CHECKS["01-infer"] = True
print("실습 1 확인 통과")
print("출력 크기:", probabilities.shape, "· 확률 합계:", f"{float(probabilities.sum()):.6f}")
print("GPU:", probabilities.device, "· 기울기 계산:", probabilities.requires_grad)
```

### 맞는 모양뿐 아니라 실제 계산 과정도 확인합니다

1. 첫 부분은 통과 표시와 이전 결과를 지웁니다. 검사가 중간에 실패했는데 예전 결과가 성공처럼 남는 일을 막습니다.
2. `callable(globals().get("infer_probabilities"))`는 그 이름으로 호출할 수 있는 함수가 메모리에 등록됐는지 봅니다. 코드가 파일에 있어도 해당 함수 셀을 아직 실행하지 않았다면 여기서 멈춥니다.
3. `weights_before`에는 실행 전 모델 지문을 저장합니다. 이어지는 `model.train()`은 **제공 함수가 평가 모드로 전환하는지 시험하기 위한 설정**입니다. 이 호출만으로 가중치가 학습되거나 바뀌지는 않습니다.
4. `record_forward`는 모델이 실제 계산을 마칠 때 호출할 관찰 함수입니다. `register_forward_hook`으로 연결해 logits와 당시의 기울기·추론·평가 모드를 기록합니다.
5. `try: ... finally: hook.remove()`는 제공 함수에서 오류가 나더라도 관찰 장치를 떼도록 합니다. 뒤의 실행에 관찰 기록이 계속 쌓이지 않게 하기 위해서입니다.
6. `assert forward_calls`는 실제 모델을 한 번 이상 호출했는지 봅니다. 숫자 1,000개를 임의로 만들면 모양이 맞아도 통과하지 못합니다.
7. 모드 검사는 평가 모드·추론 모드 안에서 기울기 없이 계산했는지 확인합니다. 다음으로 반환값이 PyTorch 텐서인지, 1차원인지, 길이가 분류 수와 같은지 검사합니다.
8. 장치 비교는 확률 텐서도 모델과 같은 GPU에 있는지 확인합니다. `next(model.parameters())`는 모델 가중치 하나를 꺼내 장치를 확인하는 데 사용합니다.
9. `torch.isfinite`는 NaN·무한대 여부를 확인합니다. 범위와 합계 검사는 유효한 확률 모양인지 봅니다. 부동소수점 오차를 고려해 합계는 정확히 `== 1` 대신 허용 오차로 비교합니다.
10. `torch.testing.assert_close`는 반환값이 실제 logits에 softmax를 적용한 결과와 가까운지 확인합니다. 마지막 모델 지문 비교는 추론 중 가중치를 바꾸지 않았는지 검사합니다.
11. 모든 검사를 지난 뒤에만 `probabilities = candidate`로 결과를 확정하고 `"01-infer"`를 `True`로 바꿉니다. `실습 1 확인 통과`와 크기·합계·GPU·기울기 상태가 표시됩니다.
12. 통과는 **함수가 이 입력에서 요구한 방식으로 추론했다는 의미**입니다. 사진의 물체를 정확히 맞혔다는 뜻이 아니며 모델의 현장 정확도를 측정한 것도 아닙니다.

## 10. 완성 함수 2 — 숫자에 이름 붙이기

원본의 20번째 셀

원본에 제공된 함수입니다. 아래 해설과 함께 코드를 읽고 원본 노트북에서 실행합니다.

```python
def topk_predictions(probabilities, id2label, k):
    values, indices = probabilities.topk(k)
    return [
        {"label": id2label[int(index)], "probability": float(value)}
        for value, index in zip(values, indices)
    ]
```

### 완성된 Top-k 함수의 값과 위치를 함께 읽기

1. `topk_predictions(probabilities, id2label, k)`는 이미 계산한 확률, 모델의 이름 연결표, 표시할 개수를 받습니다. 모델에 사진을 다시 넣거나 가중치를 학습하지 않습니다.
2. `probabilities.topk(k)`는 높은 점수부터 `k`개를 골라 **값 `values`와 위치 번호 `indices`**를 함께 돌려줍니다. 값만 고르면 어느 이름의 점수였는지 알 수 없으므로 두 결과가 모두 필요합니다.
3. `zip(values, indices)`는 같은 순위의 점수와 번호를 묶습니다. 반복문에서 `value, index`로 두 값을 나눠 받습니다.
4. `int(index)`는 텐서 안의 번호를 Python 정수로 읽습니다. `id2label[int(index)]`로 ImageNet의 예측 이름을 찾습니다. 실습 정답 다섯 개를 담은 `classes`로 찾는 코드가 아닙니다.
5. `float(value)`는 해당 점수를 Python 실수로 바꿉니다. 0~1 값은 유지하며, `60.00%` 같은 화면 표시는 뒤에서 처리합니다.
6. 중괄호는 한 후보의 `label`, `probability`를 담는 딕셔너리입니다. 바깥 대괄호와 `for`는 후보들을 순서대로 담는 **리스트 컴프리헨션**입니다.
7. `return [...]`은 이 딕셔너리들을 담은 길이 `k`의 목록을 돌려줍니다. 원래 확률 텐서와 이름 연결표는 수정하지 않습니다.
8. `k=1`, `3`, `5`로 호출하면 표시할 후보 수가 달라집니다. 같은 사진·모델에서 후보 수만 바꾸므로 공통 후보의 점수는 그대로입니다.
9. 아래 확인 셀은 작은 CPU 예시와 실제 GPU 결과에서 길이·순서·이름·점수·입력 보존을 검사합니다. 사진이 실제로 무엇인지를 채점하는 함수는 아닙니다.

**직접 확인할 질문:** 이름과 점수가 같은 순위끼리 묶이는 줄은 어디인가요? 함수가 퍼센트 문자열 대신 실수를 돌려주는 이유는 무엇인가요? 코드를 바꿔 볼 때도 두 입력을 덮어쓰지 않도록 유지하세요.

### CPU로 살펴보기 — 번호·이름·표시 형식 구분하기

이미 선택된 후보 하나에 이름을 붙이는 연습입니다. 높은 점수를 찾는 전체 함수의 답안은 아닙니다. 아래 이름과 점수는 설명을 위해 만든 값입니다.

아래는 **개념을 설명하는 작은 예시**입니다. 실제 사진이나 모델의 추론 결과가 아닙니다.

In [3]:
toy_names = {0: "예시 라벨 A", 1: "예시 라벨 B", 2: "예시 라벨 C"}
selected_id = 1
toy_score = 0.6
selected_name = toy_names[selected_id]
print("모델 번호:", selected_id)
print("이름:", selected_name)
print("계산에 쓸 숫자:", toy_score, "· 자료형:", type(toy_score).__name__)
print("화면에 보여 줄 글:", f"{toy_score:.2%}")
assert toy_score == 0.6
assert selected_name == "예시 라벨 B"

모델 번호: 1
이름: 예시 라벨 B
계산에 쓸 숫자: 0.6 · 자료형: float
화면에 보여 줄 글: 60.00%


## 11. 실습 2 확인 셀 읽기

원본의 21번째 셀

아래 회색 상자는 원본 코드의 읽기용 복사본입니다. 이 해설 노트북에서는 실행되지 않습니다.

```python
INFERENCE_CHECKS["01-topk"] = False
globals().pop("predictions", None)
if not INFERENCE_CHECKS.get("01-infer", False):
    raise RuntimeError("먼저 실습 1의 확인 셀을 통과하세요.")
if not callable(globals().get("topk_predictions")):
    raise RuntimeError("앞의 topk_predictions 함수 셀을 먼저 실행하세요.")
example = torch.tensor([0.10, 0.60, 0.03, 0.20, 0.07])
example_labels = dict(enumerate(["라벨 A", "라벨 B", "라벨 C", "라벨 D", "라벨 E"]))
for source, names in ((example, example_labels), (probabilities, model.config.id2label.copy())):
    unchanged = source.clone()
    unchanged_names = names.copy()
    for k in (1, 3, 5):
        answer = topk_predictions(source, names, k)
        assert isinstance(answer, list) and len(answer) == k, "길이가 k인 목록을 반환하세요."
        expected_values, expected_indices = source.topk(k)
        for item, value, index in zip(answer, expected_values, expected_indices):
            assert isinstance(item, dict) and set(item) == {"label", "probability"}
            assert item["label"] == names[int(index)], "번호와 이름의 연결 또는 순서를 확인하세요."
            assert type(item["probability"]) is float, "확률을 Python float로 반환하세요."
            assert abs(item["probability"] - float(value)) < 1e-6, "확률 값을 변경하지 마세요."
        assert torch.equal(source, unchanged), "입력 확률을 변경하지 마세요."
        assert names == unchanged_names, "번호와 이름의 연결을 변경하지 마세요."
predictions = topk_predictions(probabilities, model.config.id2label.copy(), 5)
INFERENCE_CHECKS["01-topk"] = True
print("실습 2 확인 통과 · 예시 입력과 실제 GPU 결과에서 k=1, 3, 5 검증")
print("실습 데이터 정답:", classes[target_class])
for item in predictions:
    print(f"{item['probability']:6.2%}  {item['label']}")
```

### 작은 예시와 실제 GPU 결과를 모두 확인합니다

1. 이전 실습 2 성공 표시와 `predictions`를 지웁니다. 먼저 실습 1이 통과했고 `topk_predictions` 함수도 있는지 검사합니다.
2. `example`은 CPU에 만든 다섯 숫자의 PyTorch 텐서입니다. `dict(enumerate([...]))`는 이름 목록에 0부터 번호를 붙여 딕셔너리로 바꿉니다.
3. 바깥 `for source, names in (...)`는 **작은 CPU 예시**와 **실제 GPU 확률** 두 경우를 시험합니다. 튜플의 두 값을 `source`, `names`로 나눠 받는 문법입니다.
4. `.clone()`은 비교용 텐서 복사본을, `.copy()`는 이름 딕셔너리 복사본을 만듭니다. 함수가 입력을 바꾸지 않았는지 확인할 기준입니다.
5. 안쪽 반복문은 `k=1, 3, 5`를 차례로 넣습니다. 한 가지 후보 수에서만 우연히 맞는 코드를 걸러냅니다.
6. `isinstance(answer, list)`와 `len(answer) == k`는 반환값이 길이 `k`의 목록인지 확인합니다. 다음 줄은 각 항목의 키가 정확히 `label`, `probability`인지 봅니다.
7. `zip(answer, expected_values, expected_indices)`는 결과 항목·정답 점수·정답 번호를 같은 위치끼리 묶습니다. 순서가 뒤섞이면 이름과 값 비교에서 실패합니다.
8. `type(item["probability"]) is float`는 Python 실수인지 엄격하게 확인합니다. 숫자로 보여도 텐서나 문자열이면 이 조건을 만족하지 못합니다.
9. 숫자는 작은 오차 범위 안에서 비교하고 `torch.equal`과 딕셔너리 비교로 입력 보존을 확인합니다. 모델을 다시 계산하는 과정은 아니며 이미 얻은 확률을 정리하는 함수의 검사입니다.
10. 모든 검사를 지나면 실제 확률의 상위 5개를 `predictions`에 저장하고 성공 표시를 켭니다. 마지막 반복문은 이름과 퍼센트를 출력합니다.
11. `실습 데이터 정답`에는 다섯 정답 종류 중 하나가, 예측 목록에는 ImageNet 이름이 나옵니다. 두 체계가 달라 한 장의 문자열 일치만으로 5종 분류 정확도를 계산할 수 없습니다.

## 12. 상위 후보를 한 개와 세 개로 비교하기

원본의 23번째 셀

아래 회색 상자는 원본 코드의 읽기용 복사본입니다. 이 해설 노트북에서는 실행되지 않습니다.

```python
if not all(INFERENCE_CHECKS.values()):
    raise RuntimeError("두 함수의 확인 셀을 먼저 통과하세요.")
for k in (1, 3):
    print(f"\n상위 {k}개를 보여 줄 때")
    for item in topk_predictions(probabilities, model.config.id2label.copy(), k):
        print(f"{item['probability']:6.2%}  {item['label']}")
```

### 같은 확률을 다른 길이로 보여 줍니다

- `INFERENCE_CHECKS.values()`는 확인 상태의 값 두 개를 꺼냅니다. `all(...)`은 두 값이 모두 참일 때만 `True`가 됩니다.
- 하나라도 통과하지 않았다면 `RuntimeError`로 멈춥니다. 결과가 완성되지 않았는데 비교를 먼저 하는 일을 막습니다.
- 바깥 반복문 `for k in (1, 3)`은 후보 수를 1과 3으로 바꿉니다.
- f-string 안의 `\n`은 줄바꿈이며 `{k}` 자리에는 현재 숫자가 들어갑니다.
- 안쪽 반복문은 해당 `k`로 정리한 결과를 한 항목씩 출력합니다. 이 셀에서 모델에 사진을 다시 넣는 것은 아닙니다.
- `id2label.copy()`는 이름 연결표의 복사본을 전달합니다. 원본 연결표를 보존하면서 비교합니다.
- 상위 3개에 있는 1위의 점수는 상위 1개를 볼 때와 같습니다. 후보를 줄였다고 남은 점수의 합을 다시 1로 맞추지 않습니다.
- 이 셀은 전역 `predictions`를 새 값으로 바꾸지 않습니다. 뒤의 그래프와 보고서는 앞 확인 셀에서 확정한 **상위 5개**를 계속 사용합니다.

원본에서 숫자를 바꿔 본 뒤 '추론 결과가 달라졌다'와 '보여 주는 범위가 달라졌다' 중 어느 설명이 맞는지 말해 보세요.

### CPU로 살펴보기 — 확인 표시가 결과 공개를 결정하는 방식

확인 셀이 성공할 때만 그래프·보고서 단계로 넘어가는 원리를 두 개의 참·거짓 값으로 살펴봅니다. 원본의 검사나 모델 결과를 대체하지 않는 설명용 예제입니다.

아래는 **개념을 설명하는 작은 예시**입니다. 실제 사진이나 모델의 추론 결과가 아닙니다.

In [4]:
toy_checks = {"첫 문제": True, "둘째 문제": False}
print("처음 상태:", toy_checks)
print("두 조건 모두 완료:", all(toy_checks.values()))
toy_checks["둘째 문제"] = True
print("둘째 문제를 마친 뒤:", toy_checks)
print("두 조건 모두 완료:", all(toy_checks.values()))
assert all(toy_checks.values()) is True

처음 상태: {'첫 문제': True, '둘째 문제': False}
두 조건 모두 완료: False
둘째 문제를 마친 뒤: {'첫 문제': True, '둘째 문제': True}
두 조건 모두 완료: True


## 13. 사진과 예측을 나란히 그리기

원본의 25번째 셀

아래 회색 상자는 원본 코드의 읽기용 복사본입니다. 이 해설 노트북에서는 실행되지 않습니다.

```python
if not all(globals().get("INFERENCE_CHECKS", {}).get(key, False)
           for key in ("01-infer", "01-topk")):
    raise RuntimeError("두 함수의 확인 셀을 모두 통과한 뒤 결과를 확인하세요.")
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
axes[0].imshow(image)
axes[0].set_title(f"Input: {classes[target_class]} (32×32)")
axes[0].axis("off")
labels = [item["label"].split(",")[0] for item in predictions]
axes[1].barh(labels[::-1], [p["probability"] for p in predictions][::-1], color="#1a73e8")
axes[1].set_xlabel("Predicted probability")
axes[1].set_xlim(0, 1)
axes[1].set_title("Pretrained ImageNet: top 5")
fig.savefig(OUTPUT_DIR / "pretrained_top5.png", dpi=150, bbox_inches="tight")
plt.show()
```

### 그림을 읽는 순서

1. 첫 조건문은 `"01-infer"`, `"01-topk"` 두 키를 직접 확인합니다. 확인 상태가 없거나 둘 중 하나라도 거짓이면 그래프를 만들지 않습니다.
2. `globals().get("INFERENCE_CHECKS", {})`는 상태 딕셔너리가 없을 때 빈 딕셔너리를 쓰고 `.get(key, False)`는 키가 없으면 미완료로 처리합니다.
3. `plt.subplots(1, 2, ...)`는 한 줄에 그래프 자리 두 개를 만듭니다. `fig`는 전체 그림, `axes[0]`과 `axes[1]`은 왼쪽·오른쪽 자리입니다.
4. 왼쪽 `imshow(image)`는 선택한 사진을 보여 줍니다. 제목의 이름은 데이터의 실제 종류이며 원본 크기는 32×32입니다. 화면에서 크게 보이는 것과 원본 해상도는 다릅니다.
5. `.axis("off")`는 사진 옆의 숫자 축을 숨깁니다. `layout="constrained"`는 제목과 그래프가 겹치지 않도록 배치를 조절합니다.
6. `[item["label"].split(",")[0] for item in predictions]`는 이름에서 첫 쉼표 앞부분만 골라 그래프 글자를 짧게 만듭니다. 저장된 전체 이름은 바꾸지 않습니다.
7. `[::-1]`은 순서를 거꾸로 읽는 슬라이싱입니다. 이름과 확률을 함께 뒤집어 가로 막대그래프에서 가장 높은 후보가 위에 오도록 합니다.
8. `barh`는 가로 막대그래프입니다. 가로 길이가 모델의 확률 값이고 `set_xlim(0, 1)`로 눈금 범위를 고정합니다.
9. `fig.savefig(...)`는 원본 GPU 세션의 결과 폴더에 PNG 파일을 저장합니다. `plt.show()`는 같은 그림을 노트북 출력에 보여 줍니다. 원본의 이 코드를 실행해야 파일이 생깁니다.
10. 막대 5개의 합이 1보다 작아도 정상일 수 있습니다. 나머지 995개 후보에도 확률이 있기 때문입니다. 틀려 보이는 예측이 있으면 사진·후보 이름·2~3위 후보를 함께 확인하세요.

## 14. 나중에 확인할 실행 기록 남기기

원본의 27번째 셀

아래 회색 상자는 원본 코드의 읽기용 복사본입니다. 이 해설 노트북에서는 실행되지 않습니다.

```python
if not all(globals().get("INFERENCE_CHECKS", {}).get(key, False)
           for key in ("01-infer", "01-topk")):
    raise RuntimeError("두 함수의 확인 셀을 모두 통과한 뒤 결과를 확인하세요.")
inference_report = {
    "status": "completed", "platform": "gpu", "device": torch.cuda.get_device_name(0),
    "model_id": MODEL_ID, "model_revision": MODEL_REVISION,
    "download_manifest_sha256": sha256_file(MODEL_DIR / "download_manifest.json"),
    "dataset_sha256": manifest["dataset_sha256"],
    "sample_id": str(splits["test"]["ids"][image_index]),
    "ground_truth": classes[target_class], "top5": predictions,
    "versions": {name: version(name) for name in ("torch", "transformers", "huggingface-hub")},
}
(OUTPUT_DIR / "inference_report.json").write_text(
    json.dumps(inference_report, ensure_ascii=False, indent=2), encoding="utf-8",
)
print("HF_GPU_INFERENCE_COMPLETE", OUTPUT_DIR / "inference_report.json")
```

### 완료 기록은 언제 만들어지나요?

1. 그래프와 같은 완료 조건을 검사합니다. 둘 중 한 문제라도 미완료이면 보고서를 새로 쓰지 않습니다.
2. `inference_report = {...}`는 실행 정보를 모은 딕셔너리입니다. `"status": "completed"`는 두 문제의 확인 조건을 통과해 이 저장 단계에 도달했다는 표시입니다. 다른 코드 셀에 오류가 없었는지는 출력 노트북에서도 따로 확인합니다.
3. `platform`과 `device`에는 GPU 사용 여부와 실제 장치 이름을 적습니다. 이 기록으로 어느 환경에서 실행했는지 확인할 수 있습니다.
4. `model_id`, `model_revision`은 모델 이름과 버전, `download_manifest_sha256`은 모델 다운로드 기록 파일의 지문입니다.
5. `dataset_sha256`은 데이터 설명서의 클래스·분할·seed·전처리 기록으로 계산한 지문입니다. 모델과 데이터 정보를 함께 남겨야 다른 실행과 비교할 때 무엇이 달랐는지 알 수 있습니다.
6. `sample_id`는 이번에 고른 사진을 가리킵니다. `str(...)`은 JSON에 넣기 쉽도록 문자열로 바꿉니다.
7. `ground_truth`는 데이터에 기록된 실제 종류이고 `top5`는 예측한 다섯 후보입니다. 여기에도 두 이름 체계가 다르다는 점이 그대로 남습니다.
8. `versions`는 torch·transformers·huggingface-hub 버전입니다. 버전이 다른 실행을 비교할 때 참고합니다.
9. `json.dumps(..., ensure_ascii=False, indent=2)`는 딕셔너리를 읽기 쉬운 JSON 글로 바꿉니다. `ensure_ascii=False`는 한글을 그대로 남기고 `indent=2`는 들여쓰기를 두 칸씩 넣습니다.
10. `.write_text(..., encoding="utf-8")`는 결과 파일을 씁니다. 같은 이름의 파일이 있으면 덮어써집니다. 마지막 `HF_GPU_INFERENCE_COMPLETE` 출력은 완료 파일 위치를 찾는 표식입니다.
11. 이 보고서는 모델을 불러와 사진 한 장을 추론한 기록입니다. 5종 분류 전체 정확도 보고서가 아닙니다. 학습 전후 비교와 최종 평가 정확도는 02번에서 확인합니다.
12. 실패한 실행은 이전 실행의 파일을 자동으로 지우지 않습니다. 예전 보고서가 있다면 파일 존재만 보고 방금 성공했다고 판단하지 말고 이번 완료 표식과 모델·사진 기록을 확인하세요. Colab 종료 전에는 원본 안내에 따라 결과 파일을 회수합니다.

## 낯선 문법을 다시 찾을 때

| 표현 | 이 노트북에서의 뜻 | 흔한 실수 |
|---|---|---|
| `name = value` | 이름에 값을 붙입니다. | 비교할 때 쓰는 `==`와 섞어 쓰기 |
| `"cup"` | 글자 데이터인 문자열입니다. | 따옴표를 빼서 변수로 해석되게 하기 |
| `items[0]`, `items[-1]` | 첫 항목, 마지막 항목을 꺼냅니다. | 첫 항목을 1번으로 생각하기 |
| `mapping["key"]` | 딕셔너리에서 이름표로 값을 찾습니다. | 정답 클래스와 모델 이름표를 바꿔 쓰기 |
| `for ...:` | 여러 항목에 같은 작업을 반복합니다. | 콜론·다음 줄 들여쓰기 빠뜨리기 |
| `if ...:` | 조건이 참일 때만 아래 코드를 실행합니다. | 참·거짓을 문자열로 만들기 |
| `def ...:` / `return` | 함수를 만들고 계산 결과를 돌려줍니다. | `print`만 하고 반환하지 않기 |
| `.shape` / `.ndim` | 축별 크기 / 축 개수입니다. | `(1, 1000)`과 `(1000,)`를 같다고 보기 |
| `.to(device)` | 텐서나 모델을 지정한 장치로 옮깁니다. | CPU 입력과 GPU 모델을 섞기 |
| `eval()` | 모델의 평가 동작을 설정합니다. | 기울기 계산까지 자동으로 꺼진다고 생각하기 |
| `inference_mode()` | 추론 구간에서 학습용 추적을 끕니다. | `eval()`을 대신한다고 생각하기 |
| `assert` | 조건이 틀리면 그 자리에서 멈춥니다. | 오류를 없애려고 검사 줄만 지우기 |

## 스스로 설명해 보기

1. RGB 사진 한 장이 `(1, 3, 224, 224)`가 될 때 각 숫자는 무엇을 뜻하나요?
2. 실습 데이터의 정답 이름은 다섯 개인데 모델 출력은 왜 1,000개인가요?
3. `model.eval()`과 `torch.inference_mode()`가 각각 바꾸는 것은 무엇인가요?
4. `top_k`를 5에서 1로 줄이면 1위 점수가 왜 100%가 되지 않나요?
5. 점수 합이 1이고 두 검사를 통과했다면 사진의 물체를 정확히 맞혔다고 할 수 있나요?
6. 입력 사진을 바꾼 뒤에는 원본의 어느 셀부터 다시 실행해야 하나요?

<details>
<summary>생각한 뒤 핵심만 확인하기</summary>

1. 사진 수·RGB 채널 수·모델 입력의 세로·가로입니다.
2. 원래 사전학습 모델의 분류 체계가 ImageNet 1,000개이기 때문입니다. 다섯 종류로 학습하는 것은 다음 실습입니다.
3. `eval()`은 모델의 동작 모드를, `inference_mode()`는 기울기 추적 등 추론에 필요 없는 작업을 제어합니다.
4. 보여 줄 후보만 줄였으며 나머지 확률을 1위에 더한 것이 아니기 때문입니다.
5. 아닙니다. 함수의 계산·반환 조건과 예측의 정확함은 별개입니다.
6. 사진 선택 준비 셀부터 두 문제의 확인·결과 출력·저장까지 다시 실행합니다. 함수 정의를 그대로 쓸 수 있는지는 앞에서 초기화했는지 확인합니다.

</details>

## 원본 실습으로 돌아가기

- [00 — 모델과 데이터 준비](../notebooks/00_hf_download_and_data.ipynb)
- [01 — GPU 추론과 핵심 함수 두 개](../notebooks/01_gpu_inference.ipynb)
- [02 — 분류 헤드 학습과 파인튜닝](../notebooks/02_gpu_finetuning.ipynb)
- [pipeline·Colab 셸 시연 안내](../pipeline-guide.md)

이 문서는 원본의 모든 코드 셀 14개를 다룹니다. 그중 두 개는 추론과 Top-k 정리를 담당하는 완성 함수 셀입니다. 설명용 CPU 예제의 숫자는 실제 모델 예측이나 GPU 재검증 결과가 아닙니다.